In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler
from scipy.stats import t
from sklearn.pipeline import Pipeline

#Directories
os.makedirs("results_total/tables", exist_ok=True)
os.makedirs("results_total/figures", exist_ok=True)

#Data
df = pd.read_csv("telemonitoring_parkinsons_updrs.data.csv")

traditional = [
    "Jitter(%)",
    "Jitter(Abs)",
    "Jitter:RAP",
    "Jitter:PPQ5",
    "Jitter:DDP",
    "Shimmer",
    "Shimmer(dB)",
    "Shimmer:APQ3",
    "Shimmer:APQ5",
    "Shimmer:APQ11",
    "Shimmer:DDA",
    "NHR",
    "HNR",
]

clinical = ["age", "sex"]
nonlinear = ["RPDE", "DFA", "PPE"]

combined = traditional + nonlinear
clinical_combined = combined + clinical

#Target
y = df["total_UPDRS"]
groups = df["subject#"]

#Target Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df["total_UPDRS"], bins=30, kde=True)
plt.xlabel("Total UPDRS")
plt.ylabel("Number of Recordings")
plt.title("Distribution of Parkinson's Severity")
plt.tight_layout()
plt.savefig("results_total/figures/updrs_distribution.png", dpi=300)
plt.close()


#Feature Sets
feature_sets = {
    "Traditional": traditional,
    "Nonlinear": nonlinear,
    "Acoustics": combined,
    "Clinical (Age and Sex)": clinical,
    "Acoustics + Clinical (Age and Sex)": clinical_combined
}


#Models
models = {
    "Linear Regression": (LinearRegression(), True),
    "Random Forest": (RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1), False),
    "Gradient Boosting": (GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42), False),
}


#Cross Validation + 95% CI
def cross_validate_model(model, features, scale=False):
    X = df[features]
    cv = GroupKFold(n_splits=5)
    if scale:
        estimator = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])
    else:
        estimator = model
    scores = cross_validate(
        estimator,
        X,
        y,
        groups=groups,
        cv=cv,
        scoring={
            "r2": "r2",
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error"
        },
        n_jobs=-1
    )
    r2_scores = scores["test_r2"]
    mae_scores = -scores["test_mae"]
    rmse_scores = -scores["test_rmse"]
    n = len(r2_scores)
    t_critical = t.ppf(
        0.975,
        df=n - 1
    )

    def calculate_stats(scores):
        mean = np.mean(scores)
        std = np.std(scores, ddof=1)
        margin = (
            t_critical
            * std
            / np.sqrt(n))

        return {
            "Mean": mean,
            "SD": std,
            "CI Lower": mean - margin,
            "CI Upper": mean + margin
        }

    r2 = calculate_stats(r2_scores)
    mae = calculate_stats(mae_scores)
    rmse = calculate_stats(rmse_scores)

    return {
        "R² Mean": r2["Mean"],
        "R² SD": r2["SD"],
        "R² 95% CI Lower": r2["CI Lower"],
        "R² 95% CI Upper": r2["CI Upper"],

        "MAE Mean": mae["Mean"],
        "MAE SD": mae["SD"],
        "MAE 95% CI Lower": mae["CI Lower"],
        "MAE 95% CI Upper": mae["CI Upper"],

        "RMSE Mean": rmse["Mean"],
        "RMSE SD": rmse["SD"],
        "RMSE 95% CI Lower": rmse["CI Lower"],
        "RMSE 95% CI Upper": rmse["CI Upper"]
    }


#Model Evaluation
cv_results = {}

for model_name, (model_obj, scale_flag) in models.items():
    model_results = {}
    for feature_name, feature_columns in feature_sets.items():
        model_results[feature_name] = cross_validate_model(
            model_obj,
            feature_columns,
            scale=scale_flag
        )
    cv_results[model_name] = pd.DataFrame(
        model_results
    ).T
cv_results = pd.concat(cv_results)


#Results Table
all_results = (
    cv_results
    .reset_index()
    .rename(columns={
        "level_0": "Model",
        "level_1": "Feature Set"
    })
)

#Create Mean ± SD columns
all_results["MAE (Mean ± SD)"] = (
    all_results["MAE Mean"].round(2).astype(str)
    + " ± "
    + all_results["MAE SD"].round(2).astype(str)
)

all_results["RMSE (Mean ± SD)"] = (
    all_results["RMSE Mean"].round(2).astype(str)
    + " ± "
    + all_results["RMSE SD"].round(2).astype(str)
)

all_results["R² (Mean ± SD)"] = (
    all_results["R² Mean"].round(3).astype(str)
    + " ± "
    + all_results["R² SD"].round(3).astype(str)
)


#Feature Importance
rf_model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)

rf_model.fit(df[combined], y)

importance = (
    pd.Series(
        rf_model.feature_importances_,
        index=combined
    )
    .sort_values()
)

plt.figure(figsize=(8, 6))
importance.plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("results_total/figures/feature_importance.png", dpi=300)
plt.close()


#Plots
#Correlation Heatmap
corr = df[combined + ["total_UPDRS"]].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Between Acoustic Biomarkers and Total UPDRS")
plt.tight_layout()
plt.savefig("results_total/figures/correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()


#Model Comparison Graph
plot_data = all_results.copy()
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_data, x="Feature Set", y="R² Mean", hue="Model")
plt.axhline(0, linestyle="--")
plt.title("R² Comparison Across Machine Learning Models")
plt.ylabel("Mean R²")
plt.xlabel("Biomarker Feature Set")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("results_total/figures/model_comparison.png", dpi=300, bbox_inches="tight")
plt.close()


#Random Forest + Acoustics
rf_predictions = cross_val_predict(rf_model, df[combined], y, groups=groups, cv=GroupKFold(n_splits=5), n_jobs=-1)
plt.figure(figsize=(6, 6))
plt.scatter(y, rf_predictions, alpha=0.6)
plt.grid(alpha=0.3)
plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    "r--",
    lw=2
)

plt.xlabel("Actual Total UPDRS")
plt.ylabel("Predicted Total UPDRS")
plt.title("Actual vs Predicted Total UPDRS (Random Forest)")
plt.tight_layout()
plt.savefig("results_total/figures/actual_vs_predicted.png",dpi=300)
plt.close()


#Error Distribution Plot
errors = y - rf_predictions
plt.figure(figsize=(8, 5))
sns.histplot(errors, bins=30, kde=True)
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Prediction Error Distribution")
plt.tight_layout()
plt.savefig("results_total/figures/error_distribution.png", dpi=300)
plt.close()


#Save
linear_results = all_results[all_results["Model"] == "Linear Regression"]
rf_results = all_results[all_results["Model"] == "Random Forest"]
gb_results = all_results[all_results["Model"] == "Gradient Boosting"]
linear_results.to_csv("results_total/tables/linear_results.csv", index=False)
rf_results.to_csv("results_total/tables/random_forest_results.csv", index=False)
gb_results.to_csv("results_total/tables/gradient_boosting_results.csv", index=False)
all_results.to_csv("results_total/tables/all_model_results.csv", index=False)
cv_results.to_csv("results_total/tables/cross_validation_results.csv")
importance.to_csv("results_total/tables/feature_importance.csv")
corr.to_csv("results_total/tables/correlations.csv")